jadi aku mau buat cleaning data, dengan skema

# Melakukan Text Preprocessing

Menghasilkan dua keluaran, pertama yaitu keluaran penghapusan tanda baca berlebih dan huruf penekanan, selanjutnya tidak adanya penghapusan tanda baca berlebih dan huruf penekanan.



Cara Kerja:

1. Case Folding

2. Penghapusan URL

3. Penghapusan Emotican

4. Menghapus White Space Berlebih



Sebagai percobaan jadi nanti ada 2 jalur, yang dibersihkan dan tidak di bersihkan.



Jalur pertama setelah itu adalah



5. Melakukan normalisasi dari kamus 1_kata_alay_gabungan.csv (dengan colomn slang, baku)

6. Normalisasi kata yang mengalami perulangan huruf berlebih (misalnya, banyakkkk menjadi banyak), dengan membatasi perulangan huruf maksimum sebanyak dua kali.

7. Penghapusan tanda baca berlebih (misalnya, titik, koma, tanda seru, atau tanda tanya yang berulang lebih dari satu kali).

8. Disimpan di result dengan nama folder hasil_normalisasi_biasa





Jalur kedua yaitu



5. Melakukan normalisasi dari kamus 2_kata_alay__penekanan.csv

6. Disimpan di result dengan nama folder hasil_normalisasi_penekanan



kalo bisa hapus white spacenya di akhir aja



hanya rubah kolomn content di csv ini path_data_review = '/data/dataset_majority_voting.csv'





In [1]:
import pandas as pd
import os
import re
import string

In [8]:
# 1. Definisi Path Utama
path_data_review = 'data/dataset_majority_voting.csv'
path_kamus_1 = 'kamus_alay/1_kata_alay_gabungan.csv'
path_kamus_2 = 'kamus_alay/2_kata_alay_penekanan.csv'

# Definisikan folder output
folder_hasil_1 = os.path.join('result')
folder_hasil_2 = os.path.join('result')

# Load Data
df = pd.read_csv(path_data_review)

# Load Kamus ke Dictionary
def load_dict(path):
    d = pd.read_csv(path)
    return dict(zip(d['slang'].astype(str), d['baku'].astype(str)))

slang_map_1 = load_dict(path_kamus_1)
slang_map_2 = load_dict(path_kamus_2)

# ==========================================
# FUNGSI PREPROCESSING DASAR
# ==========================================

def normalize_custom_patterns(text):
    if pd.isna(text): return ""
    # Jam 21:00 -> 21.00
    text = re.sub(r'(\d{2}):(\d{2})', r'\1.\2', text)
    # Angka 1.5 jam -> 1,5 jam
    text = re.sub(r'(\d+)\.(\d+)\s*(jam)', r'\1,\2 \3', text)
    return text

def basic_preprocessing(text):
    if pd.isna(text): return ""
    # 1. Case Folding
    text = text.lower()

    # Tambahan: Normalisasi pola custom (jam & angka) sebelum hapus tanda baca
    text = normalize_custom_patterns(text)

    # 2. Penghapusan URL
    text = re.sub(r'http\S+|www\S+|https\S+', '', text)
    
    # 3. Penghapusan Emoticon (ASCII Only)
    text = text.encode('ascii', 'ignore').decode('ascii')
    
    # 4. Hapus tanda petik
    text = re.sub(r"[\"'`‘’“”]", "", text)
    return text

def normalize_with_dict(text, slang_dict):
    words = text.split()
    return ' '.join([slang_dict.get(word, word) for word in words])

# ==========================================
# JALUR 1: HASIL NORMALISASI BIASA
# ==========================================
def processing_jalur_1(text):
    text = basic_preprocessing(text)

    # 5. Normalisasi kamus 1
    text = normalize_with_dict(text, slang_map_1)

    # 6. Perulangan huruf berlebih (max 2 kali)
    text = re.sub(r'([a-zA-Z])\1{2,}', r'\1\1', text)

    # 7. Penghapusan tanda baca berlebih (>1 jadi 1)
    # Note: Karena jam sudah jadi 21.00, regex ini tidak akan merusak titik tunggal
    text = re.sub(r'([!?.,])\1+', r'\1', text)

    # 4 & Akhir: Hapus White Space Berlebih
    text = re.sub(r'\s+', ' ', text).strip()
    return text

# ==========================================
# JALUR 2: HASIL NORMALISASI PENEKANAN
# ==========================================
def processing_jalur_2(text):
    text = basic_preprocessing(text)

    # 5. Normalisasi kamus 2
    text = normalize_with_dict(text, slang_map_2)

    # 4 & Akhir: Hapus White Space Berlebih
    text = re.sub(r'\s+', ' ', text).strip()
    return text

# ==========================================
# EKSEKUSI & PENYIMPANAN
# ==========================================

os.makedirs(folder_hasil_1, exist_ok=True)
os.makedirs(folder_hasil_2, exist_ok=True)

# Jalur 1
df_jalur_1 = df.copy()
df_jalur_1['content'] = df_jalur_1['content'].apply(processing_jalur_1)
df_jalur_1.to_csv(os.path.join(folder_hasil_1, 'hasil_normalisasi_biasa.csv'), index=False)

# Jalur 2
df_jalur_2 = df.copy()
df_jalur_2['content'] = df_jalur_2['content'].apply(processing_jalur_2)
df_jalur_2.to_csv(os.path.join(folder_hasil_2, 'hasil_normalisasi_penekanan.csv'), index=False)

print("Berhasil! Semua fungsi termasuk pola custom (jam/angka) sudah masuk ke kedua jalur.")

Berhasil! Semua fungsi termasuk pola custom (jam/angka) sudah masuk ke kedua jalur.
